# CO-SPY Kaggle Training

Notebook này tự clone repository, cài dependency, kiểm tra GPU, liên kết dataset và huấn luyện. Sau khi import vào Kaggle:

1. Thêm dataset **Co-Spy_v1** bằng **Add Input**.
2. Giữ **Accelerator = GPU T4 x2** và bật **Internet**.
3. Chọn **Run All**. Notebook tự chạy tuần tự nếu chỉ có một GPU.

Checkpoint được giữ trong `/kaggle/working/cospy/ckpt`; feature cache không được đưa vào file ZIP cuối cùng.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_root = Path("/kaggle/working/cospy")

if not (repo_root / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/datnguyentrung/CO-SPY.git",
            str(repo_root),
        ],
        check=True,
    )

os.chdir(repo_root)
print(f"[SETUP] Working directory: {Path.cwd()}", flush=True)

In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys

packages = [
    "diffusers==0.32.2",
    "open-clip-torch==2.31.0",
    "timm==1.0.15",
    "loguru==0.7.3",
    "accelerate==1.0.1",
    "ftfy==6.2.3",
    "tqdm==4.67.1",
    "transformers==4.46.3",
    "safetensors==0.5.3",
]

print("[SETUP] Installing CO-SPY dependencies...", flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

expected_versions = {
    "transformers": "4.46.3",
    "safetensors": "0.5.3",
}
for package, expected in expected_versions.items():
    actual = importlib.metadata.version(package)
    if actual != expected:
        raise RuntimeError(
            f"Expected {package}=={expected}, found {package}=={actual}. "
            "Restart the Kaggle session and rerun this cell."
        )

importlib.import_module("diffusers")
importlib.import_module("open_clip")
from diffusers import StableDiffusionPipeline

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Kaggle, enable Accelerator > GPU T4 x2 "
        "and Internet, then run Restart & Run All."
    )

print("[SETUP] Dependency import check passed.", flush=True)
print(
    f"[SETUP] CUDA GPUs available: {torch.cuda.device_count()}",
    flush=True,
)
print("[SETUP] Dependencies ready.", flush=True)

In [ ]:
from pathlib import Path
import os

input_root = Path("/kaggle/input")
default_dataset_root = Path(
    "/kaggle/input/datasets/datnguyentrung/co-spy-v1"
)
override_root = os.environ.get("CO_SPY_DATASET_ROOT", "").strip()

search_roots = []
for candidate in (
    Path(override_root) if override_root else None,
    default_dataset_root,
    input_root,
):
    if candidate is not None and candidate.is_dir():
        if all(candidate != existing for existing in search_roots):
            search_roots.append(candidate)


def find_directory(search_root, target_name, max_depth=8):
    search_root = Path(search_root)
    target_key = target_name.casefold()
    direct = search_root / target_name
    if direct.is_dir():
        return direct

    for current, dir_names, _ in os.walk(
        search_root,
        followlinks=False,
    ):
        current_path = Path(current)
        depth = len(current_path.relative_to(search_root).parts)
        dir_names.sort(key=str.casefold)
        for dir_name in dir_names:
            if dir_name.casefold() == target_key:
                return current_path / dir_name
        if depth >= max_depth:
            dir_names[:] = []

    raise FileNotFoundError(
        f"Cannot find directory {target_name!r} below {search_root}"
    )


def locate_source_dir(target_name):
    for search_root in search_roots:
        try:
            found = find_directory(search_root, target_name)
        except FileNotFoundError:
            continue
        print(
            f"[DATA] {target_name} source: {found}",
            flush=True,
        )
        return found

    checked = "\n".join(f"  - {root}" for root in search_roots)
    try:
        top_level = "\n".join(
            f"  - {path}" for path in sorted(input_root.iterdir())
        )
    except OSError as exc:
        top_level = f"  <cannot list {input_root}: {exc}>"
    raise FileNotFoundError(
        f"Cannot find {target_name!r} under Kaggle input.\n"
        f"Checked:\n{checked or '  <no input roots found>'}\n"
        f"Top-level {input_root}:\n{top_level}"
    )


real_source_root = locate_source_dir("mscoco2017")
fake_source_root = locate_source_dir("stable-diffusion-v1-4")

print(
    "[DATA] Dataset roots located successfully.",
    flush=True,
)

working_root = Path.cwd()
sd_root = working_root / "data" / "train" / "sd-v1_4"
real_target_root = sd_root / "mscoco2017"
fake_target_root = sd_root / "stable-diffusion-v1-4"
real_target_root.mkdir(parents=True, exist_ok=True)
fake_target_root.mkdir(parents=True, exist_ok=True)


def find_split_dir(source_parent: Path, split: str) -> Path:
    names = (f"{split}2017", split)
    matches = []

    for name in names:
        matches.extend(
            path
            for path in source_parent.rglob(name)
            if path.is_dir()
        )

    if not matches:
        raise FileNotFoundError(
            f"Cannot find {names[0]} or {names[1]} under {source_parent}"
        )

    return sorted(matches)[0]


def link_split(source_parent: Path, target_parent: Path, split: str) -> Path:
    expected = f"{split}2017"
    source_dir = find_split_dir(source_parent, split)
    target = target_parent / expected

    if target.is_symlink():
        target.unlink()
    elif target.is_dir():
        print(f"[DATA] Keeping existing directory: {target}", flush=True)
        return target

    target.symlink_to(source_dir, target_is_directory=True)
    print(
        f"[DATA] Linked {target} -> {source_dir}",
        flush=True,
    )
    return target


linked_paths = {
    "SD train": link_split(
        fake_source_root,
        fake_target_root,
        "train",
    ),
    "SD val": link_split(
        fake_source_root,
        fake_target_root,
        "val",
    ),
    "COCO train": link_split(
        real_source_root,
        real_target_root,
        "train",
    ),
    "COCO val": link_split(
        real_source_root,
        real_target_root,
        "val",
    ),
}

image_extensions = {".png", ".jpg", ".jpeg"}

for label, image_dir in linked_paths.items():
    image_count = sum(
        1
        for path in image_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    print(
        f"[DATA] {label:<12} {image_count:>8} images | {image_dir}",
        flush=True,
    )
    if image_count == 0:
        raise RuntimeError(f"No images found in {image_dir}")

print(
    "[DATA] Ready. No download or extraction was performed.",
    flush=True,
)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import threading

REPO_ROOT = Path("/kaggle/working/cospy")
PRINT_LOCK = threading.Lock()


def _stream_output(tag, process, log_path):
    with log_path.open("a", encoding="utf-8", buffering=1) as log_file:
        for raw_line in process.stdout:
            line = raw_line.rstrip("\n")
            log_file.write(raw_line)
            log_file.flush()

            with PRINT_LOCK:
                print(f"[{tag}] {line}", flush=True)


def start_job(tag, log_name, args):
    log_path = REPO_ROOT / log_name
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    command = [sys.executable, "-u", "main.py", *args]

    print(f"[{tag}] START: {' '.join(command)}", flush=True)

    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    reader = threading.Thread(
        target=_stream_output,
        args=(tag, process, log_path),
        daemon=True,
    )
    reader.start()
    return process, reader


def run_job(tag, log_name, args):
    process, reader = start_job(tag, log_name, args)
    return_code = process.wait()
    reader.join()

    print(f"[{tag}] EXIT CODE: {return_code}", flush=True)
    if return_code != 0:
        raise RuntimeError(f"{tag} failed with exit code {return_code}")

In [ ]:
import torch

artifact_args = [
    "--phase", "train",
    "--gpu", "0",
    "--mode", "branch",
    "--branch", "artifact",
    "--train_dataset", "sd-v1_4",
    "--epochs", "10",
    "--early-stopping-patience", "5",
]

semantic_args = [
    "--phase", "train",
    "--gpu", "1",
    "--mode", "branch",
    "--branch", "semantic",
    "--train_dataset", "sd-v1_4",
    "--epochs", "10",
    "--early-stopping-patience", "5",
    "--feat_interp",
]

if torch.cuda.device_count() >= 2:
    print(
        "[TRAIN] Running Artifact on GPU 0 and Semantic on GPU 1.",
        flush=True,
    )
    artifact_process, artifact_reader = start_job(
        "ART",
        "train_artifact.log",
        artifact_args,
    )
    semantic_process, semantic_reader = start_job(
        "SEM",
        "train_semantic.log",
        semantic_args,
    )

    artifact_code = artifact_process.wait()
    semantic_code = semantic_process.wait()
    artifact_reader.join()
    semantic_reader.join()

    print(
        f"[TRAIN] Artifact exit={artifact_code} | "
        f"Semantic exit={semantic_code}",
        flush=True,
    )
    if artifact_code != 0 or semantic_code != 0:
        raise RuntimeError(
            "At least one branch failed; check the logs above."
        )
else:
    print(
        "[TRAIN] One GPU detected; running the branches sequentially.",
        flush=True,
    )
    semantic_args[semantic_args.index("1")] = "0"
    run_job("ART", "train_artifact.log", artifact_args)
    run_job("SEM", "train_semantic.log", semantic_args)

In [ ]:
run_job(
    "NPR",
    "train_npr.log",
    [
        "--phase", "train",
        "--gpu", "0",
        "--mode", "branch",
        "--branch", "npr",
        "--train_dataset", "sd-v1_4",
        "--epochs", "10",
        "--early-stopping-patience", "5",
    ],
)

In [ ]:
run_job(
    "FUS2",
    "train_fusion_v2.log",
    [
        "--phase", "train",
        "--gpu", "0",
        "--mode", "fusion",
        "--train_dataset", "sd-v1_4",
        "--epochs", "20",
        "--early-stopping-patience", "5",
        "--fusion_head", "v2",
        "--artifact_views", "1",
    ],
)

In [ ]:
run_job(
    "FUS3",
    "train_fusion_v3.log",
    [
        "--phase", "train",
        "--gpu", "0",
        "--mode", "fusion",
        "--train_dataset", "sd-v1_4",
        "--epochs", "20",
        "--early-stopping-patience", "5",
        "--fusion_head", "v3",
        "--artifact_views", "1",
    ],
)

In [ ]:
run_job(
    "ART",
    "train_artifact_resume.log",
    [
        "--phase", "train",
        "--gpu", "0",
        "--mode", "branch",
        "--branch", "artifact",
        "--train_dataset", "sd-v1_4",
        "--epochs", "10",
        "--early-stopping-patience", "5",
        "--resume-mode", "auto",
    ],
)

In [ ]:
from pathlib import Path

checkpoint_root = Path("ckpt")
checkpoint_files = sorted(
    path
    for path in checkpoint_root.rglob("*")
    if path.is_file()
    and (
        path.name.startswith("checkpoint_")
        or path.name == "training_state.json"
    )
)

for path in checkpoint_files:
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"[CKPT] {path} | {size_mb:.2f} MB", flush=True)

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

checkpoint_root = Path("ckpt")
archive_path = Path("cospy_checkpoints.zip")

with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in checkpoint_root.rglob("*"):
        if not path.is_file():
            continue
        if "features" in path.parts:
            continue
        archive.write(path, path.as_posix())

print(
    f"[PACK] Created {archive_path} "
    f"({archive_path.stat().st_size / (1024 ** 3):.2f} GB)",
    flush=True,
)